 Refresh Fact Table (REPLACE - Final)

In [0]:
-- Refresh Gold Fact Table with ALL data (Day 1 + Day 2 + Day 3 + Day 4)
TRUNCATE TABLE nestle_dev_gold.bi_core.f_sales_fact;

INSERT INTO nestle_dev_gold.bi_core.f_sales_fact
SELECT 
  src.transaction_id,
  src.product_id,
  src.region,
  src.channel,
  src.customer_id,
  src.quantity,
  src.unit_price,
  src.amount,
  CAST(src.dbt_valid_from AS DATE) as transaction_date,
  CAST(src.dbt_valid_from AS TIMESTAMP) as created_at
FROM nestle_dev_silver.core.sales_transactions_scd2 src
WHERE src.dbt_is_current = TRUE;

SELECT 
  COUNT(*) as fact_rows,
  COUNT(DISTINCT product_id) as unique_products,
  COUNT(DISTINCT region) as unique_regions,
  COUNT(DISTINCT channel) as unique_channels,
  SUM(amount) as total_amount
FROM nestle_dev_gold.bi_core.f_sales_fact;

Refresh MV 1 - Daily Sales Summary

In [0]:
-- Refresh Daily Sales Summary
REFRESH MATERIALIZED VIEW nestle_dev_gold.bi_core.agg_daily_sales_summary;

SELECT 
  COUNT(*) as daily_agg_rows,
  COUNT(DISTINCT transaction_date) as unique_dates,
  COUNT(DISTINCT region) as regions_in_agg,
  SUM(total_amount) as total_agg_amount
FROM nestle_dev_gold.bi_core.agg_daily_sales_summary;

 Refresh MV 2 - Monthly Revenue by Channel

In [0]:
-- Refresh Monthly Revenue by Channel
REFRESH MATERIALIZED VIEW nestle_dev_gold.bi_core.agg_monthly_revenue_by_channel;

SELECT 
  COUNT(*) as monthly_agg_rows,
  COUNT(DISTINCT year) as years,
  COUNT(DISTINCT month) as months,
  COUNT(DISTINCT channel) as channels,
  SUM(total_revenue) as total_monthly_revenue
FROM nestle_dev_gold.bi_core.agg_monthly_revenue_by_channel;

Refresh MV 3 - Product Performance

In [0]:
-- Refresh Product Performance
REFRESH MATERIALIZED VIEW nestle_dev_gold.bi_core.agg_product_performance;

SELECT 
  COUNT(*) as product_perf_rows,
  COUNT(DISTINCT product_id) as products_tracked,
  SUM(transaction_count) as total_transactions,
  SUM(total_revenue) as total_revenue,
  AVG(avg_transaction_value) as avg_value_per_product
FROM nestle_dev_gold.bi_core.agg_product_performance;

Final Gold Layer Verification

In [0]:
SELECT 
  'Fact Table' as object_type,
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.f_sales_fact) as row_count,
  (SELECT SUM(amount) FROM nestle_dev_gold.bi_core.f_sales_fact) as total_amount,
  current_timestamp() as last_refresh

UNION ALL

SELECT 
  'Daily Sales MV' as object_type,
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.agg_daily_sales_summary) as row_count,
  NULL as total_amount,
  current_timestamp() as last_refresh

UNION ALL

SELECT 
  'Monthly Revenue MV' as object_type,
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.agg_monthly_revenue_by_channel) as row_count,
  (SELECT SUM(total_revenue) FROM nestle_dev_gold.bi_core.agg_monthly_revenue_by_channel) as total_amount,
  current_timestamp() as last_refresh

UNION ALL

SELECT 
  'Product Performance MV' as object_type,
  (SELECT COUNT(*) FROM nestle_dev_gold.bi_core.agg_product_performance) as row_count,
  (SELECT SUM(total_revenue) FROM nestle_dev_gold.bi_core.agg_product_performance) as total_amount,
  current_timestamp() as last_refresh;